In [24]:
from tf.app import use
from tf.lib import writeList

In [3]:
ORG = "HuygensING"
REPO = "suriano"

In [4]:
A = use(f"{ORG}/{REPO}:clone", checkout="clone", silent="verbose", hoist=globals())

**Locating corpus resources ...**

This is Text-Fabric 12.0.5
36 features found and 0 ignored
  0.22s Dataset without structure sections in otext:no structure functions in the T-API
  0.63s All features loaded/computed - for details use TF.isLoaded()
  0.07s All additional features loaded - for details use TF.isLoaded()


Name,# of nodes,# slots/node,% coverage
folder,2,140024.00,100
file,130,2154.22,100
body,130,2093.27,97
text,130,2093.27,97
div,756,696.80,188
table,18,144.61,1
teiHeader,130,60.95,3
chunk,6812,40.85,99
fileDesc,130,48.32,2
p,6680,39.72,95


## Sanity check

Can we reduce the entities to their tokens in stead of to their atomic tokens?

In [13]:
equal = 0
unequal = 0

for e in F.otype.s("ent"):
    tkStr = "".join((F.str.v(t) or "") + (F.after.v(t) or "") for t in L.d(e, otype="t"))
    tokenStr = "".join(F.str.v(t) + F.after.v(t) for t in L.d(e, otype="token"))
    tkStrTrim = tkStr.replace("\n", "").replace(" ", "")
    tokenStrTrim = tokenStr.replace("\n", "").replace(" ", "")
    if tkStrTrim == tokenStrTrim:
        equal += 1
    else:
        unequal += 1
        print(f"{tkStr=} {tokenStr=}")

print(f"{equal=} {unequal=}")

equal=16660 unequal=0


## Lemma sentence index

In [14]:
lemmaSentence = {}

for s in F.otype.s("sentence"):
    for x in L.d(s, otype="token"):
        lemma = F.lemma.v(x)
        if lemma:
            lemmaSentence.setdefault(lemma, []).append(s)

len(lemmaSentence)

12876

## Entity start-end index

In [16]:
eStart = {}
eEnd = {}

for e in F.otype.s("ent"):
    tokens = L.d(e, otype="token")
    eStart[tokens[0]] = e
    eEnd[tokens[-1]] = e

In [21]:
def interl(s):
    repLine = []
    lemLine = []
    entLine = []
    chunkLength = 0

    inEnt = False
    ebRep = ""
    eeRep = ""

    for t in L.d(s, otype="token"):
        after = F.after.v(t).replace("\n", " ")
        rep = F.str.v(t).replace("\u200b", "")
        lem = F.lemma.v(t).replace(" ", "_")
        lRep = len(rep)
        lLem = len(lem)
        lT = max((lRep, lLem))
        eb = eStart.get(t, None)
        ee = eEnd.get(t, None)
        ebRep = ""
        if eb:
            kind = F.kind.v(eb)
            ebRep = f"<-{kind}-"
            if not ee:
                inEnt = True
        if ee:
            if eb == ee:
                eeRep = "->"
            else:
                kind = F.kind.v(ee)
                eeRep = f"-{kind}->"
            inEnt = False
        if not eb and not ee:
            eRep = ("-" if inEnt else " ") * lT
        elif eb and not ee:
            lT = max((lT, len(ebRep)))
            eRep = ebRep.ljust(lT, "-")
        elif not eb and ee:
            lT = max((lT, len(eeRep)))
            eRep = eeRep.rjust(lT, "-")
        else:
            lT = max((lT, len(ebRep) + len(eeRep)))
            eRep = ebRep.ljust(lT - len(eeRep), "-") + eeRep
        lAfter = len(after)
        repLine.append(f"{rep.ljust(lT)}{after}")
        lemLine.append(f"{lem.ljust(lT)}{after}")
        entLine.append(f"{eRep.ljust(lT)}{after}")
        chunkLength += lAfter + lT

        if chunkLength > 100:
            output(repLine, lemLine, entLine)
            chunkLength = 0

    output(repLine, lemLine, entLine)

def output(repLine, lemLine, entLine):
    print("".join(repLine))
    print("".join(lemLine))
    print("".join(entLine))
    print("")
    repLine.clear()
    lemLine.clear()
    entLine.clear()

In [9]:
start = 85
amount = 4

i = 0

for s in F.otype.s("sentence")[start:start + amount]:
    i += 1
    print(f"---{i:>4}---")
    interl(s)


---   1---
in quel   mentre m’      arrivò   il corriero  straordinario Zupponi , che per haver  fatto il viaggio 
in quello mentre m’      arrivare il corriero￮ straordinario zupponi , che per havere fare  il viaggio 
                 <-LOC-->                                    <-PER-->                                  

per la via    de’ Grisoni, et Svizzeri, et per Lorena  , come più lunga, che quella d’Augusta  non arrivò   
per il via    de’ grisoni, et svizzeri, et per lorena  , come più lungo, che quello diaugusta  non arrivare 
       <-LOC- --- --LOC->     <-LOC-->         <-LOC-->                               <-LOC-->              

che lunedì passato, portandomi   le risolute commissioni della Serenità vostra nelle sue de’ 25. 
che lunedì passato, portandomare il risoluto commissione di_il serenità vostro in_il suo de’ 25. 
                                                               <-LOC-->                          

---   2---
Ero già innanti colla trattatione la quale

In [ ]:
lemma = "suriano"

for s in lemmaSentence[lemma]:
    interl(s)

Christofforo Suriano
christofforo suriano
<-LOC------- --LOC->

Christofforo Suriano
christofforo suriano
<-PER------- --PER->

Ernesto di Nassau li giorni passati fatto offerta al   clarissimo signor  Christofforo Suriano secretario 
ernesto di nassau li giorno passato fare  offrire a_il clarissimo signore christofforo suriano secretario 
<-PER-- -- -PER->                                                         <-PER------- --PER->            

della serenissima republica di Venetia  residente per essa appresso   li signori Stati  Generali dello 
di_il serenissima republico di venetia  residente per esso appruevere li signore stato  generali di_il 
      <-LOC----->              <-LOC-->                                          <-LOC- ---LOC->       

Provincie Unite de’ Paesi Bassi  di andar  in persona al   servitio  di detta serenissima Republica colla 
provincie unite de’ paesi bassi  di andare in persona a_il servitioo di detta sereno      republica co_il 
<-ORG---- ----- --- ---

In [ ]:
query = """
p
  sentence
    ent kind*
"""

results = A.search(query)

In [ ]:
A.show(results, end=10, condensed=True, condenseType="sentence", withNodes=True)

In [ ]:
query = """
token str~xxx
"""

results = A.search(query)

In [ ]:
query = """
ent str~xxx
"""

results = A.search(query)

In [36]:
def exportNer():
    fStr = F.str.v
    fAfter = F.after.v
    fLemma = F.lemma.v
    fKind = F.kind.v

    sentence = {}
    for s in F.otype.s("sentence"):
        for t in L.d(s, otype="token"):
            sentence[t] = s

    tokens = [(t, sentence.get(t, None), fStr(t), fAfter(t), fLemma(t)) for t in F.otype.s("token")]

    entities = []
    for e in F.otype.s("ent"):
        ts = L.d(e, otype="token")
        start = ts[0]
        end = ts[-1]
        entities.append((e, start, end, fKind(e)))
    
    destDir = f"{A.tempDir}/ner"
    writeList(tokens, f"{destDir}/tokens.csv", intCols=(True, True, False, False, False))
    writeList(entities, f"{destDir}/entities.csv", intCols=(True, True, True, False))

In [37]:
exportNer()